# 🔧 Data Preprocessing
### ML Challenge — IEEE SB, GEHU

**What we're doing:** Cleaning up the training data so it's ready to use.  
**Input:** `TRAIN.csv` (raw) → **Output:** `TRAIN_PREPROCESSED.csv` (cleaned)

**Steps:**
1. Load data & check it
2. Remove duplicates
3. Split features & target
4. Handle outliers
5. Fix skewed distributions
6. Remove correlated features
7. Scale features
8. Compare before/after
9. Save cleaned data

In [ ]:
# importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import PowerTransformer, StandardScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully')

## 📂 Step 1 — Load Data

In [ ]:
# loading the training data
df_raw = pd.read_csv('TRAIN.csv')
print(f'Dataset shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'Features: F01-F47 (47 numerical features)')
print(f'Target: Class (0=Normal, 1=Faulty)')
print(f'Missing values: {df_raw.isnull().sum().sum()}')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print(f'\nClass Distribution:')
print(df_raw['Class'].value_counts().to_string())
print(f'\nFirst 3 rows:')
df_raw.head(3)

## 🗑️ Step 2 — Remove Duplicates
Found **738 duplicate rows** in the data. These don't help and can mess with the model, so we're removing them.

In [ ]:
# removing duplicate rows
before = len(df_raw)
df = df_raw.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f'Removed {before - after} duplicate rows')
print(f'Before: {before} rows -> After: {after} rows')
print(f'Remaining duplicates: {df.duplicated().sum()}')

# checking class distribution
print(f'\nClass Distribution after removing duplicates:')
counts = df['Class'].value_counts()
for cls in sorted(counts.index):
    pct = counts[cls] / len(df) * 100
    print(f'Class {cls}: {counts[cls]} ({pct:.1f}%)')

## ✂️ Step 3 — Split Features & Target
Splitting the data into X (features) and y (target). Also keeping a copy of the original to compare later.

In [ ]:
# separating features and target
features = [c for c in df.columns if c != 'Class']
X = df[features].copy()
y = df['Class'].copy()

# keeping original data for comparison
X_original = X.copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nFeature statistics (before transformation):')
print(X.describe().T[['mean', 'std', 'min', 'max']].round(4).head(10))

## 📦 Step 4 — Handle Outliers
A lot of features (especially F30–F38) have **extreme outliers**. Capping them at the **1st and 99th percentiles** so the extreme values don't mess up training.

> Using percentile capping instead of deleting rows so we keep all the data.

In [ ]:
# capping outliers at 1st and 99th percentile

outlier_report = []

for feat in features:
    p1 = X[feat].quantile(0.01)
    p99 = X[feat].quantile(0.99)
    
    # counting outliers
    n_low = (X[feat] < p1).sum()
    n_high = (X[feat] > p99).sum()
    total_capped = n_low + n_high
    
    if total_capped > 0:
        outlier_report.append({
            'Feature': feat,
            'Low_Cap': round(p1, 4),
            'High_Cap': round(p99, 4),
            'Capped_Low': n_low,
            'Capped_High': n_high,
            'Total_Capped': total_capped,
            'Pct_Capped': round(total_capped / len(X) * 100, 2)
        })
    
    # applying capping
    X[feat] = X[feat].clip(lower=p1, upper=p99)

outlier_df = pd.DataFrame(outlier_report)
print(f'Outlier Capping Summary:')
print(f'Features with capped values: {len(outlier_df)}/{len(features)}')
print(f'Total values capped: {outlier_df["Total_Capped"].sum()}')
print(f'\nTop 10 features by capped values:')
outlier_df.sort_values('Total_Capped', ascending=False).head(10)

## 📐 Step 5 — Fix Skewed Data
**44 out of 47 features are really skewed**. Using Yeo-Johnson transform to make them more normal-looking. It works with both positive and negative values.

> Helps models like Logistic Regression, SVM, and KNN work better.

In [ ]:
# checking skewness before transformation
skew_before = X.skew()
n_skewed_before = (skew_before.abs() > 1).sum()
print(f'Skewness BEFORE:')
print(f'Highly skewed features (|skew| > 1): {n_skewed_before}/{len(features)}')
print(f'Max absolute skew: {skew_before.abs().max():.2f} ({skew_before.abs().idxmax()})')

# applying Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson', standardize=False)
X_transformed = pd.DataFrame(
    pt.fit_transform(X),
    columns=features,
    index=X.index
)

# checking skewness after transformation
skew_after = X_transformed.skew()
n_skewed_after = (skew_after.abs() > 1).sum()
print(f'\nSkewness AFTER:')
print(f'Highly skewed features (|skew| > 1): {n_skewed_after}/{len(features)}')
print(f'Max absolute skew: {skew_after.abs().max():.2f} ({skew_after.abs().idxmax()})')
print(f'\nReduced from {n_skewed_before} to {n_skewed_after} highly skewed features')

# updating X
X = X_transformed.copy()

# comparing before and after
comparison = pd.DataFrame({
    'Skew_Before': skew_before.round(3),
    'Skew_After': skew_after.round(3),
    'Improvement': (skew_before.abs() - skew_after.abs()).round(3)
}).sort_values('Improvement', ascending=False)
print(f'\nTop 10 most improved features:')
comparison.head(10)

In [ ]:
# visualization: skewness before vs after
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# before
sorted_before = skew_before.sort_values(key=abs, ascending=False)
colors_b = ['#e74c3c' if abs(v) > 1 else '#2ecc71' for v in sorted_before]
axes[0].barh(sorted_before.index, sorted_before.values, color=colors_b, edgecolor='black', linewidth=0.3)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].axvline(-1, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].axvline(1, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_title('BEFORE (Raw Data)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Skewness')
axes[0].invert_yaxis()

# after
sorted_after = skew_after.reindex(sorted_before.index)
colors_a = ['#e74c3c' if abs(v) > 1 else '#2ecc71' for v in sorted_after]
axes[1].barh(sorted_after.index, sorted_after.values, color=colors_a, edgecolor='black', linewidth=0.3)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].axvline(-1, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].axvline(1, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_title('AFTER (Yeo-Johnson)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Skewness')
axes[1].invert_yaxis()

fig.suptitle('Skewness Comparison - Before vs After', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 🔗 Step 6 — Remove Correlated Features
Found **10 feature pairs with correlation > 0.9** — basically mirror images (F01↔F29, F02↔F22, etc.).

Keeping both doesn't add anything useful and just makes things more complicated. Dropping the one that's less correlated with the target from each pair.

In [ ]:
# finding highly correlated feature pairs
corr_matrix = X.corr().abs()

# getting upper triangle
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# finding pairs with correlation > 0.9
high_corr_pairs = []
for col in upper.columns:
    for idx in upper.index:
        if upper.loc[idx, col] > 0.9:
            high_corr_pairs.append((idx, col, upper.loc[idx, col]))

print(f'Found {len(high_corr_pairs)} feature pairs with correlation > 0.9\n')

# calculating correlation with target
target_corr = {}
for feat in features:
    if feat in X.columns:
        target_corr[feat] = abs(X[feat].corr(y))

features_to_drop = set()
pair_table = []

for f1, f2, r in high_corr_pairs:
    c1 = target_corr.get(f1, 0)
    c2 = target_corr.get(f2, 0)
    drop = f2 if c1 >= c2 else f1
    keep = f1 if c1 >= c2 else f2
    features_to_drop.add(drop)
    pair_table.append({
        'Feature_A': f1, 'Corr_Target_A': round(c1, 4),
        'Feature_B': f2, 'Corr_Target_B': round(c2, 4),
        'Inter_Corr': round(r, 4),
        'Drop': drop, 'Keep': keep
    })
    print(f'{f1} (r={c1:.4f}) <-> {f2} (r={c2:.4f}) | corr={r:.3f} -> DROP {drop}')

print(f'\nFeatures to drop: {sorted(features_to_drop)}')
print(f'Dropping {len(features_to_drop)} features, keeping {len(X.columns) - len(features_to_drop)}')

# dropping redundant features
X = X.drop(columns=list(features_to_drop))
print(f'\nX shape after dropping: {X.shape}')

## ⚖️ Step 7 — Scale Features
Features are all over the place (some near 0, others in thousands). StandardScaler brings everything to **mean=0, std=1**.

> Needed for SVM, KNN, Logistic Regression, and Neural Networks. Tree models don't really need it but doesn't hurt.

In [ ]:
# scaling features using StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print('Scaling Summary:')
print(f'Method: StandardScaler (mean=0, std=1)')
print(f'Features scaled: {X_scaled.shape[1]}')
print(f'\nAfter scaling - checking mean and std:')
check = pd.DataFrame({
    'Mean': X_scaled.mean().round(6),
    'Std': X_scaled.std().round(6),
    'Min': X_scaled.min().round(4),
    'Max': X_scaled.max().round(4)
})
print(check.head(10))
print(f'\nAll means near 0: {(X_scaled.mean().abs() < 0.001).all()}')
print(f'All stds near 1: {((X_scaled.std() - 1).abs() < 0.01).all()}')

# updating X
X = X_scaled.copy()

## 📊 Step 8 — Before vs After
Comparing the **raw data** vs **cleaned data** for some key features to see what changed.

In [ ]:
# selecting features to visualize
show_feats = ['F01', 'F09', 'F19', 'F05', 'F07', 'F30']
show_feats = [f for f in show_feats if f in X.columns]

fig, axes = plt.subplots(len(show_feats), 2, figsize=(14, len(show_feats) * 2.5))

for i, feat in enumerate(show_feats):
    # before
    axes[i, 0].hist(X_original[feat], bins=50, color='#3498db', alpha=0.7, edgecolor='white', linewidth=0.3)
    axes[i, 0].set_title(f'{feat} - BEFORE (Raw)', fontsize=10, fontweight='bold')
    axes[i, 0].set_ylabel('Count')
    skew_b = X_original[feat].skew()
    axes[i, 0].text(0.98, 0.85, f'skew={skew_b:.2f}', transform=axes[i, 0].transAxes,
                    ha='right', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # after
    axes[i, 1].hist(X[feat], bins=50, color='#2ecc71', alpha=0.7, edgecolor='white', linewidth=0.3)
    axes[i, 1].set_title(f'{feat} - AFTER (Preprocessed)', fontsize=10, fontweight='bold')
    skew_a = X[feat].skew()
    axes[i, 1].text(0.98, 0.85, f'skew={skew_a:.2f}', transform=axes[i, 1].transAxes,
                    ha='right', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

fig.suptitle('Before vs After Preprocessing', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
corr_final = X.corr()
mask = np.triu(np.ones_like(corr_final, dtype=bool))
sns.heatmap(corr_final, mask=mask, cmap='coolwarm', center=0, linewidths=0.3,
            annot=False, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Heatmap - After Preprocessing', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# checking remaining correlations
upper_final = corr_final.abs().where(np.triu(np.ones(corr_final.shape), k=1).astype(bool))
max_corr = upper_final.max().max()
print(f'\nMax correlation remaining: {max_corr:.4f}')
remaining_high = (upper_final > 0.9).sum().sum()
print(f'Pairs with correlation > 0.9: {remaining_high}')
if remaining_high == 0:
    print('No highly correlated pairs remain')
else:
    print(f'{remaining_high} pairs still above 0.9')

## 💾 Step 9 — Save Cleaned Data
Saving the cleaned data for modeling. Also applying the **same steps** to `TEST.csv` so both datasets match.

In [ ]:
# saving preprocessed training data
df_preprocessed = X.copy()
df_preprocessed['Class'] = y.values

df_preprocessed.to_csv('TRAIN_PREPROCESSED.csv', index=False)
print(f'Saved: TRAIN_PREPROCESSED.csv')
print(f'Shape: {df_preprocessed.shape}')
print(f'Columns: {list(df_preprocessed.columns)}')

# preprocessing test data
print(f'\nPreprocessing TEST.csv with same steps')
df_test = pd.read_csv('TEST.csv')
print(f'Raw TEST shape: {df_test.shape}')

# separating ID
test_id = df_test['ID'] if 'ID' in df_test.columns else None
test_features = [c for c in df_test.columns if c not in ['ID', 'Class']]
X_test = df_test[test_features].copy()

# outlier capping
for feat in test_features:
    p1 = X_original[feat].quantile(0.01) if feat in X_original.columns else X_test[feat].quantile(0.01)
    p99 = X_original[feat].quantile(0.99) if feat in X_original.columns else X_test[feat].quantile(0.99)
    X_test[feat] = X_test[feat].clip(lower=p1, upper=p99)

# Yeo-Johnson transformation
X_test_transformed = pd.DataFrame(
    pt.transform(X_test[features]),
    columns=features,
    index=X_test.index
)

# dropping same features
X_test_transformed = X_test_transformed.drop(columns=list(features_to_drop))

# scaling
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_transformed),
    columns=X_test_transformed.columns,
    index=X_test.index
)

# adding ID back
df_test_preprocessed = X_test_scaled.copy()
if test_id is not None:
    df_test_preprocessed.insert(0, 'ID', test_id.values)

df_test_preprocessed.to_csv('TEST_PREPROCESSED.csv', index=False)
print(f'\nSaved: TEST_PREPROCESSED.csv')
print(f'Shape: {df_test_preprocessed.shape}')
print(f'Columns: {list(df_test_preprocessed.columns)}')

## 📝 Step 10 — Summary

In [ ]:
# ============================================================
#  PREPROCESSING SUMMARY
# ============================================================
print('=' * 65)
print('  PREPROCESSING SUMMARY')
print('=' * 65)

print(f'\nSTEP 1 - Loaded raw data:')
print(f'  {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

print(f'\nSTEP 2 - Removed duplicates:')
print(f'  {before - after} rows removed -> {after} rows remaining')

print(f'\nSTEP 3 - Separated features & target')

print(f'\nSTEP 4 - Outlier capping:')
print(f'  {len(outlier_df)} features had values capped')

print(f'\nSTEP 5 - Yeo-Johnson Transform:')
print(f'  Highly skewed features: {n_skewed_before} -> {n_skewed_after}')

print(f'\nSTEP 6 - Removed correlated features:')
print(f'  Dropped: {sorted(features_to_drop)}')
print(f'  Features: {len(features)} -> {X.shape[1]}')

print(f'\nSTEP 7 - StandardScaler applied')
print(f'  All features now have mean=0, std=1')

print(f'\nOUTPUT FILES:')
print(f'  TRAIN_PREPROCESSED.csv  ({df_preprocessed.shape})')
print(f'  TEST_PREPROCESSED.csv   ({df_test_preprocessed.shape})')

print(f'\n' + '=' * 65)
print('  Done - data is ready!')
print('=' * 65)